In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("csv")\
        .option("header",True)\
        .load("/Volumes/learnspark/raw/spark_volume/raw_orders/")

In [0]:
display(df)

## Query Plan

In [0]:
df.explain(True) # this will show all Query plans

In [0]:
df.explain() # this will show only the physical plan

## Query plan for repartition

In [0]:
df_repart = df.repartition(4)
df_repart.explain()

## Query plan for coalesce

In [0]:
df_coal = df_repart.coalesce(2)
df_coal.explain() # Exchange RoundRobinPartitioning(4) visible because we have taken df as df_repart

In [0]:
df_coal = df.coalesce(2)
df_coal.explain()

## Query Plan for Aggregate

In [0]:
df_group = (df.withColumn("spark_partition",spark_partition_id())\
    .groupBy("spark_partition")\
    .agg(count("*").alias("count"))\
    .orderBy("spark_partition"))
df_group.explain()

`+- HashAggregate(keys=[spark_partition#322], functions=[finalmerge_count(merge count#346L) AS count(1)#344L])
         +- Exchange hashpartitioning(spark_partition#322, 200), ENSURE_REQUIREMENTS, [plan_id=370]
            +- HashAggregate(keys=[spark_partition#322], functions=[partial_count(1) AS count#346L])`

- if you can see above code copied from cell 13, HashAggregate is done before shuffle, because sparks tries to aggregate as much as possible in same partition before shuffling data as this is expensive process.